# 전체 코드

## 라이브러리 설치 및 임포트

In [ ]:
# 필요한 라이브러리 설치
!pip install torch torchvision torchaudio
!pip install transformers
!pip install sentence-transformers
!pip install qdrant-client
!pip install langchain
!pip install langchain-community
!pip install langchain-openai
!pip install openai
!pip install scikit-learn
!pip install rank_bm25
!pip install tiktoken
!pip install numpy
!pip install tqdm
!pip install langchain-qdrant
!pip install nest-asyncio

In [ ]:
# 필요한 라이브러리 임포트
import os
from typing import List, Dict, Any, Optional, Tuple
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import ScoredPoint
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Qdrant as LangchainQdrant
from langchain.schema import Document
from langchain_openai import OpenAI
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity
import logging
import time
import numpy as np
from openai import OpenAI as OpenAIClient
from google.colab import userdata
import difflib
import datetime
import asyncio
from concurrent.futures import ThreadPoolExecutor
import sys
from langchain.retrievers import BM25Retriever
import nest_asyncio

## 기본 설정 및 초기화

In [ ]:
# nest_asyncio 적용
nest_asyncio.apply()

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# OpenAI API 키 설정
openai_api_key = userdata.get('FINAL_TEAM3')

# Qdrant 클라이언트 초기화
QDRANT_URL = "https://6e46b2c2-f28a-4f28-854d-432ab699fdfd.europe-west3-0.gcp.cloud.qdrant.io"
QDRANT_API_KEY = "u2eejPgTwIyhr7BVjFBtkjGdGYPWvzQTBkoYycErtm5cyrFjwEEH9w"
COLLECTION_NAME = "son99_d"

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# 임베딩 모델 초기화
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")

# Langchain Qdrant 초기화
vector_store = LangchainQdrant(
    client=client,
    collection_name=COLLECTION_NAME,
    embeddings=embeddings
)

# Sentence Transformer 모델
st_model = SentenceTransformer('jhgan/ko-sroberta-multitask')

# OpenAI 클라이언트 초기화
openai_client = OpenAIClient(api_key=openai_api_key)

# GPT 사용 제한 설정
MAX_GPT_USAGE = 3
gpt_usage_count = 0
last_reset_date = datetime.date.today()

# 샘플 모델 초기화
model_name = "centwon/ko-gpt-trinity-cw"
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:141: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.10

## QdrantRetriever

In [ ]:
class QdrantRetriever:
    # QdrantRetriever 클래스 초기화
    def __init__(self, client: QdrantClient, collection_name: str):
        """
        Args:
            client (QdrantClient): Qdrant 데이터베이스 연결을 위한 클라이언트 객체
            collection_name (str): 검색할 Qdrant 컬렉션의 이름
        """
        self.client = client
        self.collection_name = collection_name

    # 주어진 쿼리 벡터와 가장 유사한 문서를 Qdrant에서 검색
    def retrieve(self, query_vector: List[float], top_k: int = 20) -> List[Any]:
        """
        Args:
            query_vector (List[float]): 검색하려는 쿼리의 벡터 표현
            top_k (int): 반환할 가장 유사한 문서의 수 (기본값: 20)

        Returns:
            List[Any]: 검색된 문서들의 리스트. 각 문서는 Qdrant의 검색 결과 형식을 따름
        """
        results = self.client.search(
            collection_name=self.collection_name,
            query_vector=query_vector,
            limit=top_k
        )
        return results

## Qdrant 검색 결과 ->  Langchain의 Document 객체 리스트로 변환

In [ ]:
def create_documents_from_qdrant_results(results) -> List[Document]:
    """
    Qdrant 검색 결과를 Langchain Document 객체 리스트로 변환합니다.

    Args:
        results (List): Qdrant 검색 결과 리스트

    Returns:
        List[Document]: Langchain Document 객체의 리스트
    """
    documents = []
    for result in results:
        # Qdrant 결과의 payload에서 '답변' 필드를 추출. 없으면 'N/A' 반환
        text = result.payload.get('답변', 'N/A')

        # 메타데이터 딕셔너리 생성
        metadata = {
            'id': result.id,  # Qdrant 결과의 고유 ID
            '질병 카테고리': result.payload.get('질병_카테고리', 'N/A'),
            '질병': result.payload.get('질병', 'N/A'),
            '부서': result.payload.get('부서', 'N/A'),
            '의도': result.payload.get('의도', 'N/A'),
            'score': result.score  # Qdrant 검색 결과의 유사도 점수
        }

        # Langchain Document 객체 생성 및 리스트에 추가
        documents.append(Document(page_content=text, metadata=metadata))

    return documents

## ensemble model

In [ ]:
def ensemble_search(query: str, top_k: int = 5) -> List[Document]:
    """
    주어진 쿼리에 대해 Qdrant 벡터 검색과 BM25 검색을 결합한 앙상블 검색을 수행합니다.

    Args:
        query (str): 검색할 쿼리 문자열
        top_k (int): 반환할 최종 결과의 수 (기본값: 5)

    Returns:
        List[Document]: 앙상블 검색 결과의 Document 객체 리스트
    """
    # 문장을 벡터로 인코딩하는 모델 초기화
    encoder = SentenceTransformer("jhgan/ko-sroberta-multitask")

    # Qdrant 검색기 초기화
    qdrant_retriever = QdrantRetriever(client=client, collection_name="son99_d")

    # 쿼리를 벡터로 변환
    query_vector = encoder.encode(query).tolist()

    # Qdrant에서 벡터 검색 수행
    qdrant_results = qdrant_retriever.retrieve(query_vector, top_k=20)

    if not qdrant_results:
        logging.warning("Qdrant에서 검색 결과가 없습니다.")
        return []

    # Qdrant 결과를 Document 객체로 변환
    documents = create_documents_from_qdrant_results(qdrant_results)

    # BM25 검색기 초기화 및 검색 수행
    bm25_retriever = BM25Retriever.from_documents(documents)
    bm25_results = bm25_retriever.get_relevant_documents(query)

    if not bm25_results:
        logging.warning("BM25에서 유사한 문서가 없습니다.")
        return []

    # Qdrant와 BM25 결과 결합
    combined_results = []
    for bm25_result in bm25_results:
        matching_qdrant_result = next((doc for doc in documents if doc.page_content == bm25_result.page_content), None)
        if matching_qdrant_result:
            # Qdrant 점수와 BM25 점수를 0.5:0.5 비율로 결합
            combined_score = 0.5 * matching_qdrant_result.metadata.get('score', 0) + 0.5 * bm25_result.metadata.get('score', 1)
            matching_qdrant_result.metadata['combined_score'] = combined_score
            combined_results.append(matching_qdrant_result)

    if not combined_results:
        logging.warning("결합된 결과가 없습니다.")
        return []

    # 결합된 점수를 기준으로 결과 정렬
    combined_results.sort(key=lambda x: x.metadata['combined_score'], reverse=True)

    # 상위 k개의 결과 반환 (빈 내용 제외)
    return [result for result in combined_results[:top_k] if result.page_content and result.page_content.strip()]


## GPT 모델 사용량 체크

In [ ]:
def check_and_reset_gpt_usage():
    """
    GPT 모델 사용량을 체크하고 필요시 리셋합니다.
    이 함수는 일일 사용량 제한을 관리하기 위해 사용됩니다.
    """
    # 전역 변수를 사용하기 위한 선언
    global gpt_usage_count, last_reset_date

    # 오늘 날짜 가져오기
    today = datetime.date.today()

    # 마지막 리셋 날짜와 오늘 날짜를 비교
    if today > last_reset_date:
        # 새로운 날이 시작되었다면 사용량을 0으로 리셋
        gpt_usage_count = 0
        # 마지막 리셋 날짜를 오늘로 업데이트
        last_reset_date = today

## GPT4o-turbo model

In [ ]:
def generate_gpt4_response(query: str, context: Optional[str] = None, metadata: Optional[Dict[str, str]] = None, max_tokens: int = 300) -> str:
    """
    GPT-4 모델을 사용하여 의료 관련 질문에 대한 응답을 생성합니다.

    Args:
        query (str): 사용자의 질문
        context (Optional[str]): 질문에 관련된 추가 컨텍스트 정보
        metadata (Optional[Dict[str, str]]): 질문과 관련된 메타데이터
        max_tokens (int): 생성할 응답의 최대 토큰 수 (기본값: 300)

    Returns:
        str: 생성된 응답 또는 오류 메시지
    """
    try:
        # GPT-4 모델에 대한 시스템 메시지 설정
        system_message = (
            "You are an expert medical assistant designed to support seniors during medical emergencies "
            "while traveling. Your primary role is to provide accurate and clear medical advice based on the user's "
            "symptoms or questions about diseases. When the user asks about a disease or symptom, "
            "show the related metadata, such as the disease and intent, and then provide the best advice based on "
            "the retrieved information. Please respond in Korean and limit your response to approximately "
            f"{max_tokens} tokens."
        )

        # 사용자 메시지 구성 (질문, 컨텍스트, 메타데이터 포함)
        user_message = (
            f"User Query: {query}\n\n"
            f"Context: {context}\n\n"
            f"Metadata:\n"
            f"- Category: {metadata.get('질병 카테고리', 'N/A') if metadata else 'N/A'}\n"
            f"- Disease: {metadata.get('질병', 'N/A') if metadata else 'N/A'}\n"
            f"- Department: {metadata.get('부서', 'N/A') if metadata else 'N/A'}\n"
            f"- Intent: {metadata.get('의도', 'N/A') if metadata else 'N/A'}\n"
            f"- Score: {metadata.get('score', 'N/A') if metadata else 'N/A'}\n"
        )

        # GPT-4 모델에 요청 보내기
        response = openai_client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_message}
            ],
            max_tokens=max_tokens,
            temperature=0.3,  # 낮은 temperature로 더 일관된 응답 생성
            n=1,  # 하나의 응답만 생성
            stop=None,  # 특정 중지 토큰 없음
        )

        # 생성된 응답 추출 및 로깅
        generated_response = response.choices[0].message.content.strip()
        logging.info(f"GPT-4 응답 생성 완료: 길이={len(generated_response)}")

        return generated_response

    except Exception as e:
        # 오류 발생 시 로깅 및 오류 메시지 반환
        logging.error(f"GPT-4 응답 생성 중 오류 발생: {str(e)}", exc_info=True)
        return "죄송합니다. GPT-4 응답을 생성하는 중에 오류가 발생했습니다."

## ko-gpt-trinity-cw model
 - 우리가 파인 튜닝한 모델

In [ ]:
def generate_custom_model_response(query: str, context: Optional[str] = None, metadata: Optional[Dict[str, str]] = None) -> str:
    """
    사용자 정의 언어 모델을 사용하여 주어진 질문에 대한 응답을 생성합니다.

    Args:
        query (str): 사용자의 질문
        context (Optional[str]): 질문에 관련된 추가 컨텍스트 정보
        metadata (Optional[Dict[str, str]]): 질문과 관련된 메타데이터

    Returns:
        str: 생성된 응답 또는 오류 메시지
    """
    try:
        # 모델 로드 및 GPU/CPU 설정
        model = AutoModelForCausalLM.from_pretrained(model_name).to('cuda' if torch.cuda.is_available() else 'cpu')

        # 프롬프트 구성
        prompt = f"질문: {query}\n\n컨텍스트: {context or ''}\n\n메타데이터: {metadata or {}}\n\n답변:"

        # 토큰화
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        # 최대 응답 길이 설정 (질문 길이에 따라 동적으로 조정)
        max_length = min(150 + len(query) * 2, 500)

        # 응답 생성
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_return_sequences=1,  # 하나의 응답만 생성
            no_repeat_ngram_size=2,  # 2-gram 반복 방지
            temperature=0.7,  # 창의성과 일관성의 균형
            do_sample=True  # 샘플링 기반 생성
        )

        # 생성된 텍스트 디코딩 및 응답 부분 추출
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response.split("답변:")[-1].strip()

    except Exception as e:
        # 오류 처리 및 로깅
        logging.error(f"커스텀 모델 응답 생성 중 오류 발생: {str(e)}", exc_info=True)
        return "죄송합니다. 응답을 생성하는 중에 오류가 발생했습니다."

    finally:
        # 메모리 정리
        if 'model' in locals():
            del model
            torch.cuda.empty_cache()

## model 응답 생성 관리

In [ ]:
def generate_response(query: str, context: Optional[str] = None, metadata: Optional[Dict[str, str]] = None, max_tokens: int = 300) -> Tuple[str, str]:
    """
    사용자 쿼리에 대한 응답을 생성합니다. GPT-4 사용 횟수에 따라 적절한 모델을 선택합니다.

    Args:
        query (str): 사용자의 질문
        context (Optional[str]): 질문에 관련된 추가 컨텍스트 정보
        metadata (Optional[Dict[str, str]]): 질문과 관련된 메타데이터
        max_tokens (int): 생성할 응답의 최대 토큰 수 (기본값: 300)

    Returns:
        Tuple[str, str]: (생성된 응답, 사용된 모델 이름)
    """
    try:
        global gpt_usage_count
        check_and_reset_gpt_usage()

        if gpt_usage_count < MAX_GPT_USAGE:
            gpt_usage_count += 1
            return generate_gpt4_response(query, context, metadata, max_tokens), "GPT-4"
        else:
            return generate_custom_model_response(query, context, metadata), "Custom Model"
    except Exception as e:
        logging.error(f"응답 생성 중 오류 발생: {str(e)}", exc_info=True)
        return "죄송합니다. 응답을 생성하는 중에 오류가 발생했습니다.", "Error"

## 사용자 쿼리 처리

In [ ]:
async def process_query_async(query: str) -> Tuple[str, str, float]:
    """
    사용자 쿼리를 비동기적으로 처리하여 응답을 생성합니다.

    Args:
        query (str): 사용자의 질문

    Returns:
        Tuple[str, str, float]: (생성된 응답, 사용된 모델 이름, 처리 시간)
    """
    start_time = time.time()
    logging.info(f"처리 중인 쿼리: {query}")

    try:
        with ThreadPoolExecutor() as executor:
            ensemble_results = await asyncio.get_event_loop().run_in_executor(
                executor, ensemble_search, query, 5
            )

        if not ensemble_results:
            logging.warning("앙상블 검색 결과가 없습니다.")
            response, model = generate_response(query)
            return response, model, time.time() - start_time

        best_match = ensemble_results[0]
        context = best_match.page_content
        metadata = best_match.metadata

        if metadata.get('combined_score', 0) < 0.7:
            logging.warning("질문이 너무 모호해요. 조금만 더 질문해주실래요?")
            return "질문이 조금 모호한 것 같습니다. 더 구체적으로 질문해 주시겠어요?", "System Message", time.time() - start_time

        max_tokens = 900 if any(keyword in query.lower() for keyword in ['자세하게', '상세하게', '더 많은 정보']) else 300

        with ThreadPoolExecutor() as executor:
            response, model = await asyncio.get_event_loop().run_in_executor(
                executor, generate_response, query, context, metadata, max_tokens
            )

        processing_time = time.time() - start_time
        logging.info(f"처리 시간: {processing_time:.2f}초")

        return response, model, processing_time

    except Exception as e:
        logging.error(f"쿼리 처리 중 오류 발생: {str(e)}", exc_info=True)
        response, model = generate_response(query)
        return response, model, time.time() - start_time



In [ ]:
async def run_system():
    """
    의료 정보 시스템을 초기화하고 실행 준비 상태를 알립니다.
    """
    print("의료 정보 시스템을 초기화하는 중...")

    try:
        # Qdrant 연결 확인
        client.get_collections()
        print("Qdrant 서버에 성공적으로 연결되었습니다.")

        # 임베딩 모델 로드
        global st_model
        st_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
        print("임베딩 모델이 로드되었습니다.")

        # OpenAI API 키 확인
        if not openai_api_key:
            raise ValueError("OpenAI API 키가 설정되지 않았습니다.")
        print("OpenAI API 키가 확인되었습니다.")

        # 커스텀 모델 초기화
        global tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        print("커스텀 모델 토크나이저가 초기화되었습니다.")

        print("\n의료 정보 시스템이 성공적으로 초기화되었습니다.")
        print("시스템이 준비되었습니다. 질문을 입력해주세요.")

    except Exception as e:
        print(f"시스템 초기화 중 오류가 발생했습니다: {str(e)}")
        print("시스템을 종료합니다.")
        sys.exit(1)

## 성능 테스트

In [ ]:
async def evaluate_system():
    """
    시스템의 성능을 평가하는 함수입니다.
    여러 테스트 쿼리를 처리하고 결과를 출력합니다.
    """
    test_queries = [
        "순환기 질환인 고혈압 증상이 궁금해요", # 카테고리 + 질병 + 의도
        "고혈압 치료방법이 궁금해요", # 질병 + 의도
        "고혈압에 대해 알고 싶어요", # 질병
        "머리가 어지럽고 가슴통증이 있고 시야에 문제가 있고 코피가나", # 다 없음 증상만 -> 다시 질문 받는것을 확인 !
        "아 어지러운데 이게뭐지" # 유사하지 않음 -> db xx -> 다시 질문 받는것을 확인 !
    ]

    total_time = 0
    total_queries = len(test_queries)

    for query in test_queries:
        try:
            response, model, processing_time = await process_query_async(query)
            total_time += processing_time

            print(f"질문: {query}")
            print(f"답변 ({model}): {response}")
            print(f"처리 시간: {processing_time:.2f}초")
            print("-" * 50)

        except Exception as e:
            logging.error(f"쿼리 처리 중 오류 발생: {str(e)}")

    avg_time = total_time / total_queries if total_queries > 0 else 0
    print(f"평균 처리 시간: {avg_time:.2f}초")



In [ ]:
async def main():
    """
    메인 실행 함수입니다.
    시스템을 초기화하고 평가를 실행합니다.
    """
    await run_system()  # 시스템 초기화
    await evaluate_system()  # 시스템 평가 실행

if __name__ == "__main__":
    asyncio.run(main())

의료 정보 시스템을 초기화하는 중...
Qdrant 서버에 성공적으로 연결되었습니다.
임베딩 모델이 로드되었습니다.
OpenAI API 키가 확인되었습니다.
커스텀 모델 토크나이저가 초기화되었습니다.

의료 정보 시스템이 성공적으로 초기화되었습니다.
시스템이 준비되었습니다. 질문을 입력해주세요.
질문: 순환기 질환인 고혈압 증상이 궁금해요
답변 (Custom Model): 혈압은 혈관의 상태에 따라 달라집니다. 고혈압의 원인은 다양하며, 주로 유전적 요인과 비만이 영향을 받습니다. 고혈압 증상은 일반적인 증상과 유사할 수 있습니다. 정확한 진단을 위해서는 내과 검사가 필요합니다. 병력 청취, 신체 검
처리 시간: 8.37초
--------------------------------------------------
질문: 고혈압 치료방법이 궁금해요
답변 (Custom Model): 고혈압 치료는 주로 혈압 조절과 신장 기능 개선에 초점을 맞춥니다. 장기적으로는 뇌졸중과 심장 질환을 예방하는 데 도움이 됩니다. 약물만으로는 혈압을 조절하기 어렵기 때문에 식이요법, 운동
처리 시간: 9.30초
--------------------------------------------------
질문: 고혈압에 대해 알고 싶어요
답변 (Custom Model): 고혈압은 일반적으로 심혈관계 및 신장 질환에 의해 발생합니다. 고혈압의 정확한
처리 시간: 9.14초
--------------------------------------------------


질문: 머리가 어지럽고 가슴통증이 있고 시야에 문제가 있고 코피가나
답변 (System Message): 질문이 조금 모호한 것 같습니다. 더 구체적으로 질문해 주시겠어요?
처리 시간: 2.88초
--------------------------------------------------


질문: 아 어지러운데 이게뭐지고혈압 예방법에 대해서 자세하게 알려줘.
답변 (System Message): 질문이 조금 모호한 것 같습니다. 더 구체적으로 질문해 주시겠어요?
처리 시간: 1.75초
--------------------------------------------------
평균 처리 시간: 6.29초


실행 결과: (한 번 더 실행하여 custom model의 답변을 보려고 했습니다.)

의료 정보 시스템을 초기화하는 중...<br>
Qdrant 서버에 성공적으로 연결되었습니다.<br>
임베딩 모델이 로드되었습니다.<br>
OpenAI API 키가 확인되었습니다.<br>
커스텀 모델 토크나이저가 초기화되었습니다.<br>

의료 정보 시스템이 성공적으로 초기화되었습니다.<br>
시스템이 준비되었습니다. 질문을 입력해주세요.<br>
질문: 순환기 질환인 고혈압 증상이 궁금해요<br>
답변 (Custom Model): 혈압은 혈관의 상태에 따라 달라집니다. 고혈압의 원인은 다양하며, 주로 유전적 요인과 비만이 영향을 받습니다. 고혈압 증상은 일반적인 증상과 유사할 수 있습니다. 정확한 진단을 위해서는 내과 검사가 필요합니다. 병력 청취, 신체 검<br>
처리 시간: 8.37초
--------------------------------------------------
질문: 고혈압 치료방법이 궁금해요<br>
답변 (Custom Model): 고혈압 치료는 주로 혈압 조절과 신장 기능 개선에 초점을 맞춥니다. 장기적으로는 뇌졸중과 심장 질환을 예방하는 데 도움이 됩니다. 약물만으로는 혈압을 조절하기 어렵기 때문에 식이요법, 운동<br>
처리 시간: 9.30초
--------------------------------------------------
질문: 고혈압에 대해 알고 싶어요<br>
답변 (Custom Model): 고혈압은 일반적으로 심혈관계 및 신장 질환에 의해 발생합니다. 고혈압의 정확한<br>
처리 시간: 9.14초
--------------------------------------------------
WARNING:root:질문이 너무 모호해요. 조금만 더 질문해주실래요?<br>
질문: 머리가 어지럽고 가슴통증이 있고 시야에 문제가 있고 코피가나<br>
답변 (System Message): 질문이 조금 모호한 것 같습니다. 더 구체적으로 질문해 주시겠어요?<br>
처리 시간: 2.88초
--------------------------------------------------
WARNING:root:질문이 너무 모호해요. 조금만 더 질문해주실래요?<br>
질문: 아 어지러운데 이게뭐지고혈압 예방법에 대해서 자세하게 알려줘.<br>
답변 (System Message): 질문이 조금 모호한 것 같습니다. 더 구체적으로 질문해 주시겠어요?<br>
처리 시간: 1.75초
--------------------------------------------------
평균 처리 시간: 6.29초